# 03 · Unsupervised analysis of the mutation profiles

**Goal.** Look for structure in the 20-gene mutation profiles **without using the grade**, and only afterwards check whether that structure matches LGG / GBM. This is part of the EDA section of the reports.

| # | Section | Method | Question |
|---|---|---|---|
| 1 | Setup | — | Input matrix |
| 2 | PCA | Centred vs scaled, explained variance, biplot | How many directions summarise the profiles? Which genes define them? |
| 3 | Distance-based maps | MDS and t-SNE with Jaccard distance | Do patients form groups when only shared mutations count? |
| 4 | Choosing the number of clusters | k-means (WCV, silhouette) and Ward | Is there a natural number of groups? |
| 5 | Hierarchical clustering | Jaccard (average, complete, single) vs Ward | Does the linkage / distance choice matter? |
| 6 | Final clusters | k-means and Ward with the chosen k | What does each cluster look like? Are they stable? |
| 7 | Clusters vs grade | ARI, NMI, contingency tables | Does the unsupervised structure match the grade? |
| 8 | Summary | — | — |

Course material: Chapter 10 (PCA and clustering) and the weekly summary of 23 October 2026.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Project root (folder containing src/config.py) not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, fcluster
from IPython.display import display

from src import config, data, plotting, reporting, unsupervised as uns
from src.plotting import CLASS_COLORS, CLUSTER_COLORS, CLUSTER_MARKERS, NEUTRAL

plotting.set_style()
pd.set_option("display.precision", 3)

df = data.load_processed()
GENES = data.feature_columns("unsupervised")
X = df[GENES].values
y = df["Grade"].values
grade = df["Grade"].map(config.CLASS_LABELS).values
results = {}
print("Input matrix:", X.shape, "| patients without any mutation:", int((X.sum(axis=1) == 0).sum()))

## 2. Principal component analysis (PCA)

PCA finds orthogonal directions of maximum variance. The data must be **centred**; whether to also **scale** is a choice:
- **Centred only**: a 0/1 gene has variance p(1 − p), so frequent mutations (IDH1, TP53, ATRX) carry most of the variance.
- **Scaled** (unit variance): every gene weighs the same, so a gene mutated in 3% of patients counts as much as IDH1. With binary variables this up-weights rare mutations.

Both are computed. The number of components is read from the cumulative explained variance. The **biplot** shows the patients (scores) and the contribution of each gene (loadings, arrows; the six longest are drawn) on PC1 and PC2. The points are coloured by grade **after** the PCA, which never sees it. Many patients share exactly the same profile and would fall on the same point, so a small random jitter is added to the scores for display only.

In [ ]:
pca_c, scores_c = uns.fit_pca(X, scale=False)
pca_s, scores_s = uns.fit_pca(X, scale=True)
var_table = uns.pca_variance_table(pca_c, pca_s)
display(var_table)
reporting.save_table_tex(var_table, "unsup_pca_variance", float_format="{:.3f}")
results["pca"] = {
    "explained_centred": pca_c.explained_variance_ratio_, "explained_scaled": pca_s.explained_variance_ratio_,
    "n_components_80pct_centred": uns.n_components_for(pca_c, 0.80),
    "n_components_80pct_scaled": uns.n_components_for(pca_s, 0.80),
}
print("Components needed for 80% of the variance: centred", results["pca"]["n_components_80pct_centred"],
      "| scaled", results["pca"]["n_components_80pct_scaled"], "| out of", len(GENES))

fig, ax = plt.subplots(figsize=(6.5, 3.6))
k = np.arange(1, len(GENES) + 1)
ax.plot(k, np.cumsum(pca_c.explained_variance_ratio_), marker="o", ms=4, color=plotting.INK, label="Centred")
ax.plot(k, np.cumsum(pca_s.explained_variance_ratio_), marker="s", ms=4, ls="--", color=NEUTRAL, label="Scaled")
ax.axhline(0.8, color=NEUTRAL, lw=1, ls=":")
ax.text(len(GENES), 0.81, "80%", ha="right", fontsize=8, color=plotting.INK_SECONDARY)
ax.set_xticks(k)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Number of principal components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("PCA of the mutation profiles: explained variance")
ax.legend(loc="lower right")
plotting.save_figure(fig, "unsup_pca_variance")
plt.show()

In [ ]:
def biplot(ax, scores, pca, title, n_arrows=6):
    rng = np.random.default_rng(config.SEED)
    jitter = rng.normal(0, 0.04, scores[:, :2].shape)   # many patients share the same profile
    for label in ("LGG", "GBM"):
        m = grade == label
        ax.scatter(scores[m, 0] + jitter[m, 0], scores[m, 1] + jitter[m, 1], s=10, alpha=0.5,
                   color=CLASS_COLORS[label], label=label, edgecolor="none")
    load = pca.components_[:2].T
    # One common factor for both axes, so that the arrow directions are not distorted
    # (the 99th percentile of the scores avoids letting a few extreme patients set the scale)
    scale = 0.8 * np.percentile(np.abs(scores[:, :2]), 99) / np.abs(load).max()
    top = np.argsort(-np.linalg.norm(load, axis=1))[:n_arrows]
    for j in top:
        ax.annotate("", xy=(load[j, 0] * scale, load[j, 1] * scale), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="->", color=plotting.INK, lw=1.2), annotation_clip=False)
        ax.text(load[j, 0] * scale * 1.12, load[j, 1] * scale * 1.12, GENES[j], fontsize=8,
                color=plotting.INK, ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
    ev = pca.explained_variance_ratio_
    ax.set_xlabel(f"PC1 ({100 * ev[0]:.1f}% of variance)")
    ax.set_ylabel(f"PC2 ({100 * ev[1]:.1f}% of variance)")
    ax.set_title(title)
    ax.legend(loc="best", markerscale=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
biplot(axes[0], scores_c, pca_c, "Biplot, centred genes")
biplot(axes[1], scores_s, pca_s, "Biplot, scaled genes")
fig.tight_layout()
plotting.save_figure(fig, "unsup_pca_biplot")
plt.show()

load_c = uns.loadings_table(pca_c, GENES)
load_s = uns.loadings_table(pca_s, GENES)
loadings = pd.concat([load_c.add_suffix(" centred"), load_s.add_suffix(" scaled")], axis=1)
loadings = loadings.reindex(load_c["PC1"].abs().sort_values(ascending=False).index)
display(loadings)
reporting.save_table_tex(loadings.head(10).rename_axis("Gene").reset_index(), "unsup_pca_loadings",
                         float_format="{:+.2f}")
results["pca"]["loadings"] = loadings.to_dict(orient="index")

## 3. Distance-based maps: MDS and t-SNE with Jaccard distance

PCA uses Euclidean geometry, in which two patients **without** a mutation are considered similar because of that shared absence. With 20 sparse genes, shared absences dominate. The **Jaccard distance** only counts genes mutated in at least one of the two patients:

$$d_J(A, B) = 1 - \frac{|A \cap B|}{|A \cup B|}$$

- **Metric MDS** places the patients in 2-D trying to preserve all pairwise Jaccard distances. Kruskal's stress-1 measures how much the 2-D map distorts them (0 = perfect; around 0.2 or more = poor).
- **t-SNE** (mentioned in the course as a more powerful non-linear visualisation) preserves local neighbourhoods rather than all distances, so distances between far-apart groups in the plot are not meaningful.

Both are visualisations only, coloured by grade afterwards. Patients with identical profiles are at distance 0 and are drawn on top of each other, so dense groups look smaller than they are.

In [ ]:
D = uns.jaccard_distance(X)
print("Pairs at distance 0:", round(100 * np.mean(D[np.triu_indices_from(D, 1)] == 0), 2), "%",
      "| pairs at distance 1:", round(100 * np.mean(D[np.triu_indices_from(D, 1)] == 1), 2), "%")
mds_coords, stress = uns.mds_embedding(D)
tsne_coords = uns.tsne_embedding(D)
print(f"MDS Kruskal stress-1: {stress:.3f}")
results["jaccard"] = {"share_pairs_distance_0": float(np.mean(D[np.triu_indices_from(D, 1)] == 0)),
                      "share_pairs_distance_1": float(np.mean(D[np.triu_indices_from(D, 1)] == 1)),
                      "mds_stress1": stress}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, coords, title in ((axes[0], mds_coords, f"Metric MDS, Jaccard distance (stress-1 = {stress:.2f})"),
                          (axes[1], tsne_coords, "t-SNE, Jaccard distance")):
    for label in ("LGG", "GBM"):
        m = grade == label
        ax.scatter(coords[m, 0], coords[m, 1], s=10, alpha=0.55, color=CLASS_COLORS[label], label=label,
                   edgecolor="none")
    ax.set_title(title)
    ax.set_xlabel("Dimension 1"); ax.set_ylabel("Dimension 2")
    ax.legend(markerscale=2)
fig.tight_layout()
plotting.save_figure(fig, "unsup_mds_tsne_grade")
plt.show()

## 4. Choosing the number of clusters

**k-means** partitions the patients into k groups minimising the within-cluster variation (WCV). It starts from random centroids, so it is run 50 times and the best solution is kept (course: *run the algorithm several times*). On 0/1 data the squared Euclidean distance is the number of genes in which two patients differ, so the profiles are **not scaled** (scaling would make one mutation in a rare gene count as much as several frequent ones).

Criteria for each k = 2…8:
- **WCV** always decreases with k; an *elbow* would suggest a natural k.
- **Silhouette** (−1 to 1): how much closer each patient is to its own cluster than to the next one.
- **Agreement between two different algorithms** (k-means and Ward hierarchical clustering, measured with the adjusted Rand index, ARI): a partition that two methods with different assumptions both find is more likely to be real structure than an artefact of one algorithm.

**Rule (fixed before looking at the result):** the k with the highest k-means–Ward agreement is the main solution; the grade is **not** used for this choice. Because silhouette alone gives no clear answer (it keeps growing slowly with k), the next value k + 1 is also described as a **secondary, exploratory** solution, to see whether a larger k reveals sub-groups. It is reported as such, not as the chosen one.

In [ ]:
selection = uns.cluster_selection_table(X)
display(selection)
reporting.save_table_tex(selection, "unsup_k_selection", float_format="{:.3f}")
K = int(selection.loc[selection["ARI k-means vs Ward"].idxmax(), "k"])
print("Main solution (rule): k =", K, "| secondary solution: k =", K + 1)
results["k_selection"] = selection.to_dict(orient="records")
results["k_chosen"] = K
results["k_secondary"] = K + 1

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(selection["k"], selection["WCV (k-means inertia)"], marker="o", color=plotting.INK)
axes[0].set_title("Within-cluster variation"); axes[0].set_ylabel("WCV (k-means inertia)")
axes[1].plot(selection["k"], selection["Silhouette k-means"], marker="o", color=plotting.INK, label="k-means")
axes[1].plot(selection["k"], selection["Silhouette Ward"], marker="s", ls="--", color=NEUTRAL, label="Ward")
axes[1].set_title("Silhouette"); axes[1].set_ylabel("Mean silhouette"); axes[1].legend()
axes[2].plot(selection["k"], selection["ARI k-means vs Ward"], marker="o", color=plotting.INK)
axes[2].axvline(K, color=NEUTRAL, ls=":", lw=1)
axes[2].set_title("Agreement k-means vs Ward"); axes[2].set_ylabel("ARI")
for ax in axes:
    ax.set_xlabel("Number of clusters k")
fig.tight_layout()
plotting.save_figure(fig, "unsup_k_selection")
plt.show()

## 5. Hierarchical clustering: does the distance / linkage choice matter?

Agglomerative clustering builds a dendrogram from the leaves (patients) to the trunk. It needs (1) a **dissimilarity** between patients and (2) a **linkage** between groups: complete (largest distance), average, single (smallest distance) or, for Euclidean data, Ward (smallest increase of within-cluster variance).

The table compares Jaccard with three linkages and Euclidean with Ward, cut into the chosen k. The **cophenetic correlation** measures how faithfully the dendrogram preserves the original distances, but a high value does not guarantee useful clusters: the cluster sizes show whether the cut produces real groups or just isolates a few patients.

In [ ]:
links = uns.linkage_comparison(X, D, K)
with pd.option_context("display.max_colwidth", None):
    display(links)
reporting.save_table_tex(links, "unsup_linkage_comparison", float_format="{:.3f}", column_format="llrrlr")
results["linkage_comparison"] = links.to_dict(orient="records")

from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage
L_jac = linkage(squareform(D, checks=False), method="average")
L_ward = uns.ward_linkage(X)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, L, title in ((axes[0], L_jac, "Jaccard distance, average linkage"), (axes[1], L_ward, "Euclidean distance, Ward linkage")):
    dendrogram(L, ax=ax, truncate_mode="lastp", p=30, color_threshold=0, above_threshold_color=plotting.INK_SECONDARY,
               no_labels=True)
    ax.set_title(title); ax.set_ylabel("Merge distance"); ax.grid(axis="x", visible=False)
    ax.set_xlabel("Patients (last 30 merges)")
fig.tight_layout()
plotting.save_figure(fig, "unsup_dendrograms")
plt.show()

## 6. Final clusters

Both solutions (main k and secondary k + 1) are computed with k-means and Ward. Each k-means cluster is described by its size, its proportion of GBM, the median age and the prevalence of each mutation. Clusters are named C1, C2, … in decreasing proportion of GBM (only to give them stable names; the grade did not take part in building them).

**Stability.** k-means with a single random initialisation is compared with the best of 50 runs (ARI = 1: identical partition).

In [ ]:
SOLUTIONS = {}
for k in (K, K + 1):
    km = uns.kmeans_labels(X, k)
    SOLUTIONS[k] = {
        "kmeans": uns.order_clusters(km.labels_, y),
        "ward": uns.order_clusters(fcluster(L_ward, k, criterion="maxclust"), y),
        "stability": uns.kmeans_stability(X, k, km.labels_),
    }
    s = SOLUTIONS[k]
    print(f"--- k = {k} ---")
    print(f"Single-initialisation runs identical to the best of 50: {int(np.sum(s['stability'] > 0.999))} / {len(s['stability'])}")
    print("Agreement k-means vs Ward (ARI):", round(uns.adjusted_rand_score(s["kmeans"], s["ward"]), 3))
    display(pd.crosstab(pd.Series(s["kmeans"], name="k-means"), pd.Series(s["ward"], name="Ward")))
    prof, prevalence = uns.cluster_profiles(df, s["kmeans"], GENES)
    s["profile"], s["prevalence"] = prof, prevalence
    display(prof)
    results[f"k{k}"] = {"kmeans_stability_ari": s["stability"],
                        "kmeans_vs_ward_ari": uns.adjusted_rand_score(s["kmeans"], s["ward"]),
                        "profiles": prof.to_dict(orient="index"),
                        "prevalence": prevalence.round(2).to_dict(orient="index")}
    reporting.save_table_tex(prof.rename_axis("Cluster").reset_index(), f"unsup_cluster_profiles_k{k}", float_format="{:.1f}")

genes_shown = [g for g in GENES if max(SOLUTIONS[k]["prevalence"][g].max() for k in SOLUTIONS) >= 10]
fig, axes = plt.subplots(2, 1, figsize=(9, 4.6), gridspec_kw={"height_ratios": [K, K + 1]})
for ax, k in zip(axes, (K, K + 1)):
    sns.heatmap(SOLUTIONS[k]["prevalence"][genes_shown], annot=True, fmt=".0f", cmap=plotting.SEQUENTIAL,
                vmin=0, vmax=100, linewidths=1, linecolor="white", cbar=False, ax=ax)
    ax.set_title(f"k-means, k = {k}" + (" (main solution)" if k == K else " (secondary, exploratory)"), fontsize=10)
    ax.set_xlabel(""); ax.set_ylabel(""); ax.tick_params(axis="y", rotation=0)
axes[0].set_xticklabels([])
fig.suptitle("Mutation prevalence (%) in each cluster (genes above 10% in some cluster)", fontweight="bold", fontsize=11)
fig.tight_layout()
plotting.save_figure(fig, "unsup_cluster_profiles")
plt.show()

## 7. Clusters versus grade

The clusters are now compared with the grade, which was never used to build them:
- **Contingency table** and percentage of GBM in each cluster.
- **Adjusted Rand index (ARI)**: agreement between two partitions corrected for chance (0 = chance, 1 = identical).
- **Normalised mutual information (NMI)**: shared information between the two partitions (0 to 1).

A cluster structure does not need to coincide with the grade to be meaningful: with more clusters than classes, one grade can be split into sub-groups (which lowers the ARI with the grade even if the clusters are real). The histological diagnosis is shown only to describe the clusters (it is never a predictor).

In [ ]:
rows = []
for k in (K, K + 1):
    for method in ("kmeans", "ward"):
        rows.append({"Partition": f"{'k-means' if method == 'kmeans' else 'Ward'} (k = {k})",
                     **uns.agreement_with_grade(SOLUTIONS[k][method], y)})
agreement = pd.DataFrame(rows)
display(agreement)
reporting.save_table_tex(agreement, "unsup_agreement_grade", float_format="{:.3f}")
results["agreement_with_grade"] = agreement.to_dict(orient="records")

for k in (K, K + 1):
    labels = SOLUTIONS[k]["kmeans"]
    ct_grade = pd.crosstab(pd.Series(labels, name="Cluster"), pd.Series(grade, name="Grade"))[["LGG", "GBM"]]
    ct_dx = pd.crosstab(pd.Series(labels, name="Cluster"), df[config.MULTICLASS_TARGET].rename("Diagnosis"))
    SOLUTIONS[k]["ct_grade"] = ct_grade
    print(f"--- k = {k} ---"); display(ct_grade); display(ct_dx)
    results[f"k{k}"]["vs_grade"] = ct_grade.to_dict(orient="index")
    results[f"k{k}"]["vs_diagnosis"] = ct_dx.to_dict(orient="index")
    reporting.save_table_tex(ct_dx.rename_axis("Cluster").reset_index(), f"unsup_clusters_vs_diagnosis_k{k}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, k in zip(axes, (K, K + 1)):
    ct_grade = SOLUTIONS[k]["ct_grade"]
    share = ct_grade.div(ct_grade.sum(axis=1), axis=0)
    left = np.zeros(len(share))
    for label in ("LGG", "GBM"):
        ax.barh(share.index, share[label], left=left, color=CLASS_COLORS[label], label=label, height=0.6,
                edgecolor="white", linewidth=2)
        left += share[label].values
    for i, n in enumerate(ct_grade.sum(axis=1)):
        ax.text(1.01, i, f"n = {n}", va="center", fontsize=8, color=plotting.INK_SECONDARY)
    ax.set_xlim(0, 1.15); ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.set_xlabel("Proportion of patients")
    ax.set_title(f"Grade within each k-means cluster, k = {k}")
axes[0].legend(loc="upper center", bbox_to_anchor=(1.1, -0.18), ncol=2)
fig.tight_layout()
plotting.save_figure(fig, "unsup_clusters_vs_grade")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, k in zip(axes, (K, K + 1)):
    labels = SOLUTIONS[k]["kmeans"]
    for i, c in enumerate(sorted(set(labels))):
        m = labels == c
        ax.scatter(mds_coords[m, 0], mds_coords[m, 1], s=12, alpha=0.6, color=CLUSTER_COLORS[i],
                   marker=CLUSTER_MARKERS[i], label=c, edgecolor="none")
    ax.set_title(f"MDS map (Jaccard) coloured by k-means cluster, k = {k}")
    ax.set_xlabel("Dimension 1"); ax.set_ylabel("Dimension 2"); ax.legend(markerscale=2)
fig.tight_layout()
plotting.save_figure(fig, "unsup_mds_clusters")
plt.show()

## 8. Summary

Generated from the results above and saved in `results/logs/03_unsupervised.json`.

In [ ]:
summary_rows = [
    ("PCA", f"80% of the variance needs {results['pca']['n_components_80pct_centred']} components (centred) and "
            f"{results['pca']['n_components_80pct_scaled']} (scaled) out of {len(GENES)}; PC1 explains "
            f"{100 * pca_c.explained_variance_ratio_[0]:.1f}% (centred) vs {100 * pca_s.explained_variance_ratio_[0]:.1f}% (scaled)"),
    ("PC1 loadings (centred)", ", ".join(f"{g} {load_c.loc[g, 'PC1']:+.2f}" for g in load_c['PC1'].abs().sort_values(ascending=False).index[:4])),
    ("MDS (Jaccard)", f"stress-1 = {stress:.3f}"),
    ("Number of clusters", f"main k = {K} (highest k-means vs Ward agreement, ARI = {selection['ARI k-means vs Ward'].max():.3f}); "
                           f"secondary k = {K + 1} (ARI = {selection.set_index('k').loc[K + 1, 'ARI k-means vs Ward']:.3f})"),
    ("Hierarchical with Jaccard", "; ".join(f"{r['Linkage']}: largest cluster {r['Largest cluster (%)']:.0f}% of patients"
                                            for _, r in links[links['Distance'] == 'Jaccard'].iterrows())),
    ("k-means stability", "; ".join(f"k = {k}: {int(np.sum(SOLUTIONS[k]['stability'] > 0.999))} of "
                                    f"{len(SOLUTIONS[k]['stability'])} single-initialisation runs reach the best solution"
                                    for k in (K, K + 1))),
    ("Clusters vs grade", "; ".join(f"{r['Partition']}: ARI = {r['ARI']:.3f}, NMI = {r['NMI']:.3f}" for _, r in agreement.iterrows())),
    (f"Clusters, k = {K}", "; ".join(f"{c}: n = {int(r['n'])}, {r['GBM (%)']:.0f}% GBM, median age {r['Median age']:.0f}"
                                     for c, r in SOLUTIONS[K]['profile'].iterrows())),
    (f"Clusters, k = {K + 1}", "; ".join(f"{c}: n = {int(r['n'])}, {r['GBM (%)']:.0f}% GBM, median age {r['Median age']:.0f}"
                                         for c, r in SOLUTIONS[K + 1]['profile'].iterrows())),
]
summary = pd.DataFrame(summary_rows, columns=["Topic", "Finding"])
with pd.option_context("display.max_colwidth", None):
    display(summary)
results["summary"] = dict(summary_rows)
reporting.save_table_tex(summary, "unsup_summary", column_format="lp{12cm}")
path = reporting.save_json(results, "03_unsupervised")
print("Saved", path.relative_to(config.ROOT))
print("Figures:", sorted(p.name for p in config.FIGURES.glob("unsup_*.png")))